In [1]:
import sys

import pm4py

import pandas as pd
import numpy as np

import torch
from torch.utils.data import DataLoader

from sklearn.model_selection import train_test_split

from utils.general_utils import set_stdout_to_file, set_seed

from config.feature_config import FeatureConfig

from model.preprocessor import PreprocessorArtifacts
from model.next_event_model import ProcessLSTM, train_ProcessLSTM, validate_ProcessLSTM

### --- Preprocess Dataset ---

In [2]:
set_seed(seed=42)

In [3]:
log = pm4py.read_xes("../../data/bpic12.xes")

C:\Users\dcoralage\Downloads\counterfactual_exp\counterfactual_env\lib\site-packages\pm4py\utils.py:1027: UserWarning: Install the optional requirement `r4pm` to import/export files faster. `rustxes` remains supported as a fallback.
  warnings.warn(
C:\Users\dcoralage\Downloads\counterfactual_exp\counterfactual_env\lib\site-packages\pm4py\util\dt_parsing\parser.py:82: UserWarning: ISO8601 strings are not fully supported with strpfromiso for Python versions below 3.11
  warnings.warn(


parsing log, completed traces ::   0%|          | 0/13087 [00:00<?, ?it/s]

In [4]:
df = pm4py.convert_to_dataframe(log)

In [5]:
df = df[~df["concept:name"].str.startswith("W_", na=False)].copy()

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 92093 entries, 0 to 262198
Data columns (total 7 columns):
 #   Column                Non-Null Count  Dtype              
---  ------                --------------  -----              
 0   org:resource          92093 non-null  object             
 1   lifecycle:transition  92093 non-null  object             
 2   concept:name          92093 non-null  object             
 3   time:timestamp        92093 non-null  datetime64[ns, UTC]
 4   case:REG_DATE         92093 non-null  datetime64[ns, UTC]
 5   case:concept:name     92093 non-null  object             
 6   case:AMOUNT_REQ       92093 non-null  object             
dtypes: datetime64[ns, UTC](2), object(5)
memory usage: 5.6+ MB


In [7]:
df.isnull().any()

org:resource            False
lifecycle:transition    False
concept:name            False
time:timestamp          False
case:REG_DATE           False
case:concept:name       False
case:AMOUNT_REQ         False
dtype: bool

In [8]:
df['case:concept:name'] = df['case:concept:name'].astype('string')
df['concept:name'] = df['concept:name'].astype('string')
df['lifecycle:transition'] = df['lifecycle:transition'].astype('string')
df['org:resource'] = df['org:resource'].astype('string')

df['case:AMOUNT_REQ'] = df['case:AMOUNT_REQ'].astype(np.float32)

df['time:timestamp'] = pd.to_datetime(df['time:timestamp'], errors='coerce')
df['case:REG_DATE'] = pd.to_datetime(df['case:REG_DATE'], errors='coerce')

In [9]:
df = df.sort_values(by=['case:concept:name', 'time:timestamp'], ascending=[True, True])

In [10]:
df['time_delta'] = df.groupby('case:concept:name')['time:timestamp'].diff()
df['time_delta'] = df['time_delta'].dt.total_seconds().astype(np.float32)
df['time_delta'] = df['time_delta'].fillna(0)

In [11]:
df['case:REG_DATE_HR'] = df['case:REG_DATE'].dt.strftime('%I %p')
df['case:REG_DATE_DAY'] = df['case:REG_DATE'].dt.day_name()
df['case:REG_DATE_MON'] = df['case:REG_DATE'].dt.month_name()
df = df.drop(columns=['case:REG_DATE'])

In [12]:
exclude_cols = ["case:concept:name", "time:timestamp"]

sorted_cols = sorted(
    [c for c in df.columns if c not in exclude_cols]
)

df = df[exclude_cols + sorted_cols]

In [13]:
df.head(20)

,case:concept:name,time:timestamp,case:AMOUNT_REQ,case:REG_DATE_DAY,case:REG_DATE_HR,case:REG_DATE_MON,concept:name,lifecycle:transition,org:resource,time_delta
0,173688,2011-10-01 00:38:44.546000+00:00,20000.0,Saturday,12 AM,October,A_SUBMITTED,COMPLETE,112,0.000000
1,173688,2011-10-01 00:38:44.880000+00:00,20000.0,Saturday,12 AM,October,A_PARTLYSUBMITTED,COMPLETE,112,0.334000
2,173688,2011-10-01 00:39:37.906000+00:00,20000.0,Saturday,12 AM,October,A_PREACCEPTED,COMPLETE,112,53.026001
5,173688,2011-10-01 11:42:43.308000+00:00,20000.0,Saturday,12 AM,October,A_ACCEPTED,COMPLETE,10862,39785.402344
6,173688,2011-10-01 11:45:09.243000+00:00,20000.0,Saturday,12 AM,October,O_SELECTED,COMPLETE,10862,145.934998
7,173688,2011-10-01 11:45:09.243000+00:00,20000.0,Saturday,12 AM,October,A_FINALIZED,COMPLETE,10862,0.000000
8,173688,2011-10-01 11:45:11.197000+00:00,20000.0,Saturday,12 AM,October,O_CREATED,COMPLETE,10862,1.954000
9,173688,2011-10-01 11:45:11.380000+00:00,20000.0,Saturday,12 AM,October,O_SENT,COMPLETE,10862,0.183000
17,173688,2011-10-10 11:33:03.668000+00:00,20000.0,Saturday,12 AM,October,O_SENT_BACK,COMPLETE,11049,776872.312500
21,173688,2011-10-13 10:37:29.226000+00:00,20000.0,Saturday,12 AM,October,A_REGISTERED,COMPLETE,10629,255865.562500


In [14]:
num_cases = df['case:concept:name'].nunique()
print(f"Total number of unique cases: {num_cases}")

Total number of unique cases: 13087


### --- Feature Configurations ---

In [15]:
# --- Define feature specs ---
feature_specs = {

    "time_delta": {
        "type":           "continuous",
        "level":          "event",
        "vary":           True,
        "quantile_low":   0.20,
        "quantile_high":  0.80, 
    },

    "case:AMOUNT_REQ": {
        "type":           "continuous",
        "level":          "case",
        "vary":           True,
                 
    },

    "case:REG_DATE_DAY": {
        "type":           "categorical",
        "level":          "case",
        "vary":           True,
    },

    "case:REG_DATE_MON": {
        "type":           "categorical",
        "level":          "case",
        "vary":           True,
    },

    "case:REG_DATE_HR": {
        "type":           "categorical",
        "level":          "case",
        "vary":           True,
    },

    "org:resource": {
        "type":           "categorical",
        "level":          "event",
        "vary":           True,
    },

    # immutable
    "concept:name": {
        "type":           "categorical", 
        "level":          "event",
        "vary":           False
    },

    "lifecycle:transition": {
        "type":           "categorical", 
        "level":          "event",
        "vary":           False
    },
}

In [16]:
feature_config = FeatureConfig.from_dataframe(
    df=df,
    feature_specs=feature_specs,
    activity_feature="concept:name",
    is_robust=True,
    default_quantile_low=0.05,
    default_quantile_high=0.95
)

feature_config.save()

In [17]:
# feature_config = FeatureConfig.load()

In [18]:
feature_config.summary()

  activity_feature:    concept:name
  feature_order:       ['case:AMOUNT_REQ', 'case:REG_DATE_DAY', 'case:REG_DATE_HR', 'case:REG_DATE_MON', 'concept:name', 'lifecycle:transition', 'org:resource', 'time_delta']
--------------------------------------------------------------------------------------------------------------------------------------
Feature                        Type           Level    Vary   Range/Categories                         MAD        Source              
--------------------------------------------------------------------------------------------------------------------------------------
time_delta                     continuous     event    yes    [0.00, 1315.95]                          0.4980     quantile_derived    
case:AMOUNT_REQ                continuous     case     yes    [3000.00, 40000.00]                      5000.0000  quantile_derived    
case:REG_DATE_DAY              categorical    case     yes    ['Friday', 'Monday', 'Saturday', ...]    N/A        

### --- Next Event Prediction Model ---

In [19]:
# Transform nan cols to NA
cat_cols = df.select_dtypes(include=["string"]).columns
for col in cat_cols:
    df[col] = df[col].fillna("NA").astype('string')

In [20]:
case_ids = df["case:concept:name"].unique()

train_cases, val_cases = train_test_split(case_ids, test_size=0.2, random_state=42)

train_df = df[df["case:concept:name"].isin(train_cases)].copy()
val_df   = df[df["case:concept:name"].isin(val_cases)].copy()

In [21]:
preprocessor_artifacts = PreprocessorArtifacts.build(
    df=train_df,
    feature_config=feature_config,      
    scaler_type="robust",
)

preprocessor_artifacts.save()

In [22]:
# preprocessor_artifacts = PreprocessorArtifacts.load()

In [23]:
preprocessor_artifacts.summary()

===================PreprocessorArtifacts====================
  scaler:              RobustScaler
  encoders:            ['case:REG_DATE_DAY', 'case:REG_DATE_MON', 'case:REG_DATE_HR', 'org:resource', 'concept:name', 'lifecycle:transition']
  activity_prototypes: 17 activities


In [24]:
# Transform nan cols to 0
float_cols = df.select_dtypes(include=["float32", "float64"]).columns
df[float_cols] = df[float_cols].fillna(0)
train_df[float_cols] = train_df[float_cols].fillna(0)
val_df[float_cols] = val_df[float_cols].fillna(0)

In [25]:
train_dataset = preprocessor_artifacts.transform_dataframe_to_processdataset(
    df=train_df,
    case_id_field="case:concept:name", 
    sort_field="time:timestamp"
)

val_dataset = preprocessor_artifacts.transform_dataframe_to_processdataset(
    df=val_df,
    case_id_field="case:concept:name", 
    sort_field="time:timestamp"
)

In [26]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

In [27]:
print(preprocessor_artifacts.get_categorical_feature_cardinality())

{'dynamic_categorical_info': {'concept:name': 17, 'lifecycle:transition': 1, 'org:resource': 61}, 'static_categorical_info': {'case:REG_DATE_DAY': 7, 'case:REG_DATE_HR': 24, 'case:REG_DATE_MON': 5}}


In [28]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [29]:
device

device(type='cuda')

In [30]:
criterion = torch.nn.CrossEntropyLoss()

In [31]:
log_file, original_stdout = set_stdout_to_file(filepath="logs/bpic12-model_output.txt")

Epoch 020/100 | Train Loss: 0.5584 | LR: 9.05e-04
Epoch 040/100 | Train Loss: 0.5307 | LR: 6.55e-04
Epoch 060/100 | Train Loss: 0.5194 | LR: 3.46e-04
Epoch 080/100 | Train Loss: 0.4795 | LR: 9.64e-05
Epoch 100/100 | Train Loss: 0.4591 | LR: 1.00e-06
Time taken for next event model (training): 1316.231919 seconds
Time taken for next event model (validation): 0.870839 seconds
Val loss: {'loss': 0.47547434262580274, 'accuracy': 0.7839690850162054, 'f1_macro': 0.710430150566411, 'f1_weighted': 0.776707919480358}


In [32]:
embedding_metadata = preprocessor_artifacts.get_embedding_metadata()

model = ProcessLSTM(
    dynamic_categorical_info=embedding_metadata["dynamic_categorical_info"],
    static_categorical_info=embedding_metadata["static_categorical_info"],
    n_dynamic_continuous=embedding_metadata["n_dynamic_continuous"],
    n_static_continuous=embedding_metadata["n_static_continuous"],
    n_classes=embedding_metadata["n_classes"]
)

train_loss_history = train_ProcessLSTM(
    model=model,
    train_loader=train_loader,
    learning_rate=1e-3,
    criterion=criterion,
    num_epochs=100,
    device=device
)

model.save()

In [33]:
# model = ProcessLSTM.load()

In [34]:
val_loss = validate_ProcessLSTM(
    model=model,
    val_loader=val_loader,
    criterion=criterion,
    device=device
)

print("Val loss:", val_loss)

### --- Cleanup ---

In [35]:
# --- Save processed df ---
if df["time:timestamp"].dt.tz is not None:
    df["time:timestamp"] = df["time:timestamp"].dt.tz_convert(None)
df.to_excel("../../data/bpic12.xlsx", index=False, engine="openpyxl")

In [36]:
sys.stdout = original_stdout
log_file.close()